In [0]:
# Day 6: Medallion Architecture (Bronze -> Silver -> Gold)
# Goals:
# - Bronze: raw ingest with audit columns
# - Silver: validated + deduped + typed + feature-ready
# - Gold  : business aggregates (idempotent build)
# - Pattern: batch_id + ingestion_ts + deterministic keys (Talend-style reliability)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

# Batch metadata
batch_id = f"day06_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
ingestion_ts = F.current_timestamp()

# Source paths
OCT_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
NOV_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"

# Medallion paths (Volume-safe)
BRONZE_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events"
SILVER_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events"
GOLD_PRODUCT_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf"
GOLD_CATEGORY_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/gold/category_perf"

In [0]:
# Design 3-layer architecture
print(f"batch_id: {batch_id}")
print("Bronze =", BRONZE_PATH)
print("Silver =", SILVER_PATH)
print("Gold product =", GOLD_PRODUCT_PATH)
print("Gold category =", GOLD_CATEGORY_PATH)

In [0]:
# Bronze raw ingestion
raw_oct = spark.read.option("header", True).option("inferSchema", True).csv(OCT_PATH)
raw_nov = spark.read.option("header", True).option("inferSchema", True).csv(NOV_PATH)

raw = raw_oct.unionByName(raw_nov)

bronze = (
    raw
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingestion_ts", ingestion_ts)
)

print("Bronze rows:", bronze.count())
bronze.show(5,truncate=False)

In [0]:
# Write Bronze Delta
bronze.write.format("delta").mode("overwrite").save(BRONZE_PATH)
print("Bronze written to Delta.")

In [0]:
# Silver cleaning & validation
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

silver_typed = (
    bronze_df
    .withColumn("event_ts", F.to_timestamp(F.col("event_time")))
    .drop("event_time")
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("event_date", F.to_date("event_ts"))
)


In [0]:
# Data quality rules (minimal but meaningful)
silver_valid = (
    silver_typed
    .filter(F.col("event_ts").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("price").isNotNull())
    .filter((F.col("price") > 0) & (F.col("price") < 10000))
)

In [0]:
# Deterministic dedupe
cols = set(silver_valid.columns)

if {"user_session", "event_ts"}.issubset(cols):
    dedupe_keys = ["user_session", "event_ts"]
elif {"user_id", "product_id", "event_ts"}.issubset(cols):
    dedupe_keys = ["user_id", "product_id", "event_ts"]
else:
    raise Exception(f"No suitable dedupe keys found. Columns: {sorted(cols)}")

w = Window.partitionBy(*dedupe_keys).orderBy(F.col("ingestion_ts").desc())

silver = (
    silver_valid
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn(
        "price_tier",
        F.when(F.col("price") < 10, "budget")
         .when(F.col("price") < 50, "mid")
         .otherwise("premium")
    )
)

print("Silver rows:", silver.count())
silver.select("event_ts","event_type","product_id","brand","category_code","price","price_tier").show(5, truncate=False)


In [0]:
# Write Silver Data

silver.write.format("delta").mode("overwrite").save(SILVER_PATH)
print("Silver written to Delta.")

In [0]:
# Gold aggregates - Read Silver Data
silver_df = spark.read.format("delta").load(SILVER_PATH)

In [0]:
# Gold - Product performance (views, purchases, revenue, conversion)

product_perf = (
    silver_df
    .groupBy("product_id", "brand", "category_code")
    .agg(
        F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("views"),
        F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("purchases"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))).alias("revenue")
    )
    .withColumn("views", F.coalesce(F.col("views"), F.lit(0)))
    .withColumn("purchases", F.coalesce(F.col("purchases"), F.lit(0)))
    .withColumn("revenue", F.coalesce(F.col("revenue"), F.lit(0.0)))
    .withColumn(
        "conversion_rate",
        F.when(F.col("views") == 0, F.lit(0.0)).otherwise(F.col("purchases") / F.col("views") * 100)
    )
    .orderBy(F.desc("revenue"))
)

product_perf.show(10, truncate=False)

In [0]:
# Gold - Category performance (leader-friendly aggregate)

category_perf = (
    silver_df
    .groupBy("category_code")
    .agg(
        F.count("*").alias("events"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))).alias("revenue"),
        F.countDistinct("user_id").alias("active_users")
    )
    .withColumn("revenue", F.coalesce(F.col("revenue"), F.lit(0.0)))
    .orderBy(F.desc("revenue"))
)

category_perf.show(10, truncate=False)

In [0]:
# Write Gold Delta
product_perf.write.format("delta").mode("overwrite").save(GOLD_PRODUCT_PATH)
category_perf.write.format("delta").mode("overwrite").save(GOLD_CATEGORY_PATH)
print("Gold written to Delta.")

In [0]:
# Incremental Silver using MERGE (Optional)

from delta.tables import DeltaTable

# Create target table if missing
try:
    DeltaTable.forPath(spark, SILVER_PATH)
    target_exists = True
except:
    target_exists = False

if not target_exists:
    silver.write.format("delta").mode("overwrite").save(SILVER_PATH)

silver_dt = DeltaTable.forPath(spark, SILVER_PATH)

# Use the same key logic as dedupe_keys
cond = " AND ".join([f"t.{k} = s.{k}" for k in dedupe_keys])

(
    silver_dt.alias("t")
    .merge(silver.alias("s"), cond)
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

silver_dt.history(3).select("version","timestamp","operation").show(truncate=False)


In [0]:
# One-shot validation report
print("=== Medallion Validation Summary ===")
print("Bronze rows :", spark.read.format("delta").load(BRONZE_PATH).count())
print("Silver rows :", spark.read.format("delta").load(SILVER_PATH).count())
print("Gold products:", spark.read.format("delta").load(GOLD_PRODUCT_PATH).count())
print("Gold categories:", spark.read.format("delta").load(GOLD_CATEGORY_PATH).count())
print("Batch id:", batch_id)